In [ ]:
import pandas as pd
import numpy as np

In [ ]:
raw_df = pd.read_csv('globalterrorismdb_0718dist.csv', encoding='ISO-8859-1', low_memory=False)

In [ ]:
df = raw_df.dropna(subset=['summary']).copy()

In [ ]:
df = df.sample(n=10000, random_state=42).copy()

In [ ]:
df['iday'] = df['iday'].replace(0, 1)
df['imonth'] = df['imonth'].replace(0, 1)
df['date'] = pd.to_datetime(df[['iyear', 'imonth', 'iday']].rename(
    columns={'iyear': 'year', 'imonth': 'month', 'iday': 'day'}
))

In [ ]:
df['headline'] = df['attacktype1_txt'] + " in " + df['city'] + ", " + df['country_txt']

In [ ]:
final_schema_df = pd.DataFrame({
    'event_id': df['eventid'],
    'date': df['date'],
    'country': df['country_txt'],
    'region': df['region_txt'],
    'headline': df['headline'],
    'category': df['attacktype1_txt'], # Baseline category, NLP will improve this .
    'source': df['dbsource'],          # Where the data came from
    'raw_text': df['summary']          # The text for your NLP model
})

In [ ]:
print("\n--- Data Quality Check ---")
print("Total Rows:", len(final_schema_df))
print("Missing Values:\n", final_schema_df.isnull().sum())
print("Duplicate IDs:", final_schema_df.duplicated(subset=['event_id']).sum())

In [ ]:
final_schema_df.to_csv('cleaned_raw_events.csv', index=False)
print("\nSuccess: Cleaned data exported to 'cleaned_raw_events.csv'")

In [ ]:
pip install pandas transformers torch tqdm

In [ ]:
from transformers import pipeline
from tqdm import tqdm

In [ ]:
tqdm.pandas(desc="Classifying Events")

In [ ]:
df = pd.read_csv('cleaned_raw_events.csv')

In [ ]:
df = df.dropna(subset=['raw_text']).drop_duplicates(subset=['event_id'])

In [ ]:
df['country'] = df['country'].str.strip().str.title()

In [ ]:
print(" Extracting Date Features...")
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['week'] = df['date'].dt.isocalendar().week

In [ ]:
print("Initializing Zero-Shot NLP Model (Downloading model if first run)...")

In [ ]:
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli",device=0)

In [ ]:
candidate_labels = [
    "Military Conflict",
    "Cyberattack",
    "Terrorism",
    "Diplomatic Tension",
    "Civil Unrest"
]

In [ ]:
def categorize_text(text):
    # Truncate to ~1000 characters to prevent token-length errors in the model
    truncated_text = str(text)[:1000]
    result = classifier(truncated_text, candidate_labels)
    # The pipeline returns lists ordered by confidence score. Index [0] is the highest match.
    return result['labels'][0]

In [ ]:
print(" Running NLP Classification ...")

In [ ]:
df['nlp_category'] = df['raw_text'].progress_apply(categorize_text)

In [ ]:
print("Calculating the severity scores....")

In [ ]:
base_severity = {
    "Military Conflict": 4,
    "Terrorism": 5,
    "Civil Unrest": 3,
    "Cyberattack": 4,
    "Diplomatic Tension": 2
}

In [ ]:
df['severity_score'] = df['nlp_category'].map(base_severity)

In [ ]:
severe_keywords = ['casualty', 'dead', 'killed', 'explosion', 'critical']
def adjust_severity(row):
    text_lower = str(row['raw_text']).lower()
    if any(word in text_lower for word in severe_keywords):
        return min(row['severity_score'] + 1, 5) # Cap at 5
    return row['severity_score']

In [ ]:
df['severity_score'] = df.apply(adjust_severity, axis=1)

In [ ]:
print("6. Exporting Final Dataset...")
df.to_csv('processed_events.csv', index=False)
print("Success! Data completely transformed and saved to 'processed_events.csv'")